## Define the libaries

In [ ]:
using DifferentialEquations
using LinearAlgebra
using NPZ
using ProgressBars
using ProgressMeter
using Integrals
using Plots

## Import global values of resonator and tunneling

In [ ]:
#### transport parameters

const e = 1.602e-19             # elementary charge
const hbar=4.135667696e-15/(2*pi) # in eV Hz^{−1}

const Vb= .25# gate bias of the arches  (mV)
const alphaVg= 0.08334 # lever arm in eV/V
const EVbias = Vb ## in (meV) --> e=1 in this case 
const muR = - EVbias /2    # right chemical potential (meV)
const muL = EVbias/2   # left chemical potential (meV)

#### tunneling parameters

###################### parameters from the N-2 peak
const aGLN_2= 0.430264 ## in (GHz/2/pi)
const aGRN_2= 44.2383 ## in (GHz/2/pi)
const gmN_2= 0.8 ## in GHz/2/pi
const V0N_2=  -3.07135  ## in (V)
const gm_fm_N_2= 2.78 ## unit-less

###################### parameters from the N-1 peak
const aGLN_1=  0.737844  ## in (GHz/2/pi)
const aGRN_1= 24.0938 ## in (GHz/2/pi)
const gmN_1= 0.75 ## in GHz/2/pi
const V0N_1= -3.16863 ## in (V)
const gm_fm_N_1= 2.60 ## unit-less

###################### parameters from the N peak
const aGLN=  1.17658  ## in (GHz/2/pi)
const aGRN= 52.808 ## in (GHz/2/pi)
const gmN= 0.96 ## in GHz/2/pi
const V0N= -3.2814  ## in (V)
const gm_fm_N= 3.38 ## unit-less

###################### parameters from the N+1 peak
const aGLN1= 2.13673  ## in (GHz/2/pi)
const aGRN1= 110.354 ## in (GHz/2/pi)
const gmN1= 1.2 ## in GHz/2/pi
const V0N1= -3.35679    ## in (V)
const gm_fm_N1= 4.14 ## unit-less

###################### parameters from the N+2 peak
const aGLN2= 1.02658  ## in (GHz/2/pi)
const aGRN2= 37.7511 ## in (GHz/2/pi)
const gmN2= 0.8  ## in GHz/2/pi
const V0N2= -3.45068 ## in (V)
const gm_fm_N2= 2.78 ## unit-less


#dimentionalize all the parameters
function param_dim(aGL,aGR,gm)

    GL = aGL*2*pi*1e9## in (Hz)
    GR = aGR*2*pi*1e9## in (Hz)
    a=aGL/aGR ## unitless
    Gtot = (GL + GR) ## in (Hz)
    hGtot_meV = hbar*(GL + GR)*1e3## in meV

    g=gm*2*pi*1e9 ## in (Hz) 
    
    return GL,GR,a,Gtot,hGtot_meV,g
end

const GLN_2,GRN_2,aN_2,GtotN_2,hGtotN_2_meV,gN_2 = param_dim(aGLN_2,aGRN_2,gmN_2)
const GLN_1,GRN_1,aN_1,GtotN_1,hGtotN_1_meV,gN_1= param_dim(aGLN_1,aGRN_1,gmN_1)
const GLN,GRN,aN,GtotN,hGtotN_meV,gN = param_dim(aGLN,aGRN,gmN)
const GLN1,GRN1,aN1,GtotN1,hGtotN1_meV,gN1 = param_dim(aGLN1,aGRN1,gmN1)
const GLN2,GRN2,aN2,GtotN2,hGtotN2_meV,gN2 = param_dim(aGLN2,aGRN2,gmN2)
;

#### resonator parameters
const delta=1.15e-3/1.5
const epsilon=9.375e-9

### resonance freqeuncy of the resonator
const wm=289*2*pi # in MHz



In [ ]:
## range of frequency to sweep 
# nomralized with w0
const w1=280*2*pi/wm
const w2=305*2*pi/wm
# Frequency Points
wm_arch = LinRange(w1,w2,251) #Angular frequency 
fm_arch = LinRange(w1,w2,251).*(wm/2/pi) # frequency used in experiemnts

## range of voltage to sweep 
const V1=-3.425 ## in (V)
const V2=-3.125 ## in (V)
# Frequency Points
Vg_arch = LinRange(V1,V2,201) 


### Import solution for initial conditions in the duffing regime
Inital_cond=npzread("InitialConditions/Experimental_simulations_intial_conditions.npz")
sweep_up=Inital_cond["duffing_sweep_up"];
sweep_down=Inital_cond["duffing_sweep_down"];


## Define the duffing equaiton of motion



In [ ]:
F_values=[0.1,2,4];


steady_state_up_F=zeros(length(wm_arch),length(F_values))
steady_state_down_F=zeros(length(wm_arch),length(F_values)) 


for f in eachindex(1:length(F_values))
    @showprogress for point in eachindex(1:length(fm_arch))
        ### define the frequency to evaluate
        F0=F_values[f]
        omega=wm_arch[point]
        tspan = (0,5000*pi/omega)
        function duffing_eqn_ODE(du, u, p, t)
            du[1] = u[2]
            du[2] = - delta * u[2] -  u[1] - epsilon * u[1]^3 + F0* cos(omega * t)
        end
        ### solve the ODE problem
        ## up sweep
        u0_up = [sweep_up[point]; 0]
        prob_up = ODEProblem(duffing_eqn_ODE, u0_up, tspan)
        sol_up = solve(prob_up, Vern9(),dt=1e-10,abstol=1e-5,reltol=1e-5)
        

        ## down sweep
        u0_down = [0;0]
        prob_down = ODEProblem(duffing_eqn_ODE, u0_down, tspan)
        sol_down = solve(prob_down, Vern9(),dt=1e-10,abstol=1e-5,reltol=1e-5)
        
        ### calcualte the maximun such that average*e^{i*omega*t}
        start=800
        average_up=maximum(sol_up[1,end-30:end])
        average_down=maximum(sol_down[1,end-30:end])
    

        ### save
        steady_state_up_F[point,f]=average_up
        steady_state_down_F[point,f]=average_down

    end
end


In [ ]:
### plot the results
p1 = plot(wm_arch,[sweep_up,steady_state_up_F[:,1],steady_state_up_F[:,2],steady_state_up_F[:,3]],title="up")
p2= plot(wm_arch,[sweep_down, steady_state_down_F[:,1],steady_state_down_F[:,2],steady_state_down_F[:,3]],title="down")

plot(p1,p2,layout=(2,1),xticks = 0.98:.01:1.04)

### Incorporate the back action of the electron transport

In [ ]:

for f in eachindex(1:length(F_values))
    F0=F_values[f]
    
    ultrastrong_sweep_up=zeros(length(fm_arch),length(Vg_arch))
    ultrastrong_sweep_down=zeros(length(fm_arch),length(Vg_arch))
    
    ultrastrong_current_up=zeros(length(fm_arch),length(Vg_arch))
    ultrastrong_current_down=zeros(length(fm_arch),length(Vg_arch))


    @showprogress for point in eachindex(1:length(fm_arch))
        ## define the value of frequency
        omega=wm_arch[point]
        tspan = (0,5000*pi/omega) 
        for j in 1:length(Vg_arch)
            ## defien the gate voltage
            Vg=Vg_arch[j]
            ### solve ODE
            #################################
            function ultrastrong_ODE(du,u,p,t)
                # population of the QD-> N-2
                μN_2 = -alphaVg*(V0N_2 - Vg) + gN_2*hbar*u[1] ## in eV
                pN_2= 0.5 + (atan(2*(muR - μN_2*1e3)/hGtotN_2_meV) + aN_2*atan(2*(muL - μN_2*1e3)/hGtotN_2_meV))/(pi*(1 + aN_2))

                # population of the QD-> N-1
                μN_1 = -alphaVg*(V0N_1 - Vg)+ gN_1*hbar*u[1] ## in eV
                pN_1= 0.5 + (atan(2*(muR - μN_1*1e3)/hGtotN_1_meV) + aN_1*atan(2*(muL - μN_1*1e3)/hGtotN_1_meV))/(pi*(1 + aN_1))


                # population of the QD -> N
                μN = -alphaVg*(V0N - Vg) + gN*hbar*u[1] ## in eV
                pN= 0.5 + (atan(2*(muR - μN*1e3)/hGtotN_meV) + aN*atan(2*(muL - μN*1e3)/hGtotN_meV))/(pi*(1 + aN))


                # population of the QD-> N+1
                μN1 = -alphaVg*(V0N1 - Vg) + gN1*hbar*u[1] ## in eV
                pN1= 0.5 + (atan(2*(muR - μN1*1e3)/hGtotN1_meV) + aN1*atan(2*(muL - μN1*1e3)/hGtotN1_meV))/(pi*(1 + aN1))

                # population of the QD-> N+2
                μN2 = -alphaVg*(V0N2 - Vg) + gN2*hbar *u[1] ## in eV
                pN2= 0.5 + (atan(2*(muR - μN2*1e3)/hGtotN2_meV) + aN2*atan(2*(muL - μN2*1e3)/hGtotN2_meV))/(pi*(1 + aN2))


                # diferential equation
                du[1] = u[2]
                du[2] = - delta * u[2] -  (u[1]+2*(gm_fm_N_2*pN_2+gm_fm_N_1*pN_1+gm_fm_N*pN+gm_fm_N1*pN1+gm_fm_N2*pN2))- epsilon * u[1]^3 + F0* cos(omega * t)
            end
        #################################
            ## up sweep
            u0_up = [sweep_up[point]; 0]
            prob_up = ODEProblem(ultrastrong_ODE, u0_up, tspan)
            sol_up = solve(prob_up, Vern9(),dt=1e-10,abstol=1e-5,reltol=1e-5)

            ## down sweep
            u0_down = [0; 0]
            prob_down = ODEProblem(ultrastrong_ODE, u0_down, tspan)
            sol_down = solve(prob_down, Vern9(),dt=1e-10,abstol=1e-5,reltol=1e-5)

            ### calucalte the average
            average_up= maximum(sol_up[1,end-30:end])
            average_down= maximum(sol_down[1,end-30:end])

            ### save the results
            ultrastrong_sweep_up[point,j]=average_up
            ultrastrong_sweep_down[point,j]=average_down

            ### compute the current
            ### up sweep function
            function I_up(t,p)
                x=cos(t*omega)*average_up
                μN_2 = -alphaVg*(V0N_2 - Vg) + gN_2*hbar*x ## in eV
                IN_2=(atan(2*(muL-μN_2*1e3)/hGtotN_2_meV)-atan(2*(muR-μN_2*1e3)/hGtotN_2_meV))*(GLN_2*GRN_2)/GtotN_2
                
                μN_1 = -alphaVg*(V0N_1 - Vg)+ gN_1*hbar*x ## in eV
                IN_1=(atan(2*(muL-μN_1*1e3)/hGtotN_1_meV)-atan(2*(muR-μN_1*1e3)/hGtotN_1_meV))*(GLN_1*GRN_1)/GtotN_1

                μN = -alphaVg*(V0N - Vg) + gN*hbar*x ## in eV
                IN=(atan(2*(muL-μN*1e3)/hGtotN_meV)-atan(2*(muR-μN*1e3)/hGtotN_meV))*(GLN*GRN)/GtotN

                μN1 = -alphaVg*(V0N1 - Vg) + gN1*hbar*x ## in eV
                IN1=(atan(2*(muL-μN1*1e3)/hGtotN1_meV)-atan(2*(muR-μN1*1e3)/hGtotN1_meV))*(GLN1*GRN1)/GtotN1

                μN2 = -alphaVg*(V0N2 - Vg) + gN2*hbar*x ## in eV
                IN2=(atan(2*(muL-μN2*1e3)/hGtotN2_meV)-atan(2*(muR-μN2*1e3)/hGtotN2_meV))*(GLN2*GRN2)/GtotN2
                
                I=IN_2+IN_1+IN+IN1+IN2
            end

            domain = (0, 4*pi/omega) 
            prob_up = IntegralProblem(I_up, domain)
            sol_up = solve(prob_up, QuadGKJL())

            #### same function as before but used for the down sweep
            function I_down(t,p)
                x=cos(t*omega)*average_down
                μN_2 = -alphaVg*(V0N_2 - Vg) + gN_2*hbar*x ## in eV
                IN_2=(atan(2*(muL-μN_2*1e3)/hGtotN_2_meV)-atan(2*(muR-μN_2*1e3)/hGtotN_2_meV))*(GLN_2*GRN_2)/GtotN_2
                
                μN_1 = -alphaVg*(V0N_1 - Vg)+ gN_1*hbar*x ## in eV
                IN_1=(atan(2*(muL-μN_1*1e3)/hGtotN_1_meV)-atan(2*(muR-μN_1*1e3)/hGtotN_1_meV))*(GLN_1*GRN_1)/GtotN_1

                μN = -alphaVg*(V0N - Vg) + gN*hbar*x ## in eV
                IN=(atan(2*(muL-μN*1e3)/hGtotN_meV)-atan(2*(muR-μN*1e3)/hGtotN_meV))*(GLN*GRN)/GtotN

                μN1 = -alphaVg*(V0N1 - Vg) + gN1*hbar*x ## in eV
                IN1=(atan(2*(muL-μN1*1e3)/hGtotN1_meV)-atan(2*(muR-μN1*1e3)/hGtotN1_meV))*(GLN1*GRN1)/GtotN1

                μN2 = -alphaVg*(V0N2 - Vg) + gN2*hbar*x ## in eV
                IN2=(atan(2*(muL-μN2*1e3)/hGtotN2_meV)-atan(2*(muR-μN2*1e3)/hGtotN2_meV))*(GLN2*GRN2)/GtotN2
                
                I=IN_2+IN_1+IN+IN1+IN2
            end
            prob_down = IntegralProblem(I_down, domain)
            sol_down = solve(prob_down, QuadGKJL())

            # ### save the results
            ultrastrong_current_up[point,j]=sol_up.u/(4*pi/omega) 
            ultrastrong_current_down[point,j]=sol_down.u/(4*pi/omega) 
        end
    end
    
    ## save each file containing all the data
    dicctionary=Dict("Vg" => Vg_arch, "fm" =>fm_arch, "wm" => wm_arch,
    "displacement_up" =>ultrastrong_sweep_up, 
    "displacement_down" =>ultrastrong_sweep_down,
    "current_up" =>ultrastrong_current_up, 
    "current_down" =>ultrastrong_current_down)
    npzwrite("Simulations/Experimental_simulations_fits_F_"*string(F0)*".npz", dicctionary)
end